In [0]:
%pip install yfinance
%pip install pandas

In [0]:
import yfinance as yf
import requests
from bs4 import BeautifulSoup
import pandas as pd

from pyspark.sql.functions import (
    col,
    current_timestamp,
    expr
)

# -----------------------------------
# Scrape article content
# -----------------------------------
def scrape_article_content(url):

    if not url:
        return None

    try:
        headers = {
            "User-Agent": "Mozilla/5.0"
        }

        response = requests.get(
            url,
            headers=headers,
            timeout=10
        )

        soup = BeautifulSoup(
            response.content,
            "html.parser"
        )

        article_div = soup.find(
            "div",
            class_="caas-body"
        )

        if article_div:
            return article_div.get_text(
                separator=" "
            ).strip()

        paragraphs = soup.find_all("p")

        content = " ".join(
            p.text for p in paragraphs
        )

        return (
            content
            .replace(
                "Oops, something went wrong",
                ""
            )
            .strip()
        )

    except:
        return None


# -----------------------------------
# Download news
# -----------------------------------
ticker_symbol = "AAPL"

ticker = yf.Ticker(ticker_symbol)

news_list = ticker.news

all_data = []

if news_list:

    for item in news_list:

        content_dict = item.get("content", {})

        if not content_dict:
            continue

        title = content_dict.get(
            "title",
            "No Title"
        )

        published_at = content_dict.get(
            "displayTime"
        )

        summary = content_dict.get(
            "summary"
        )

        content_type = content_dict.get(
            "contentType"
        )

        click_url = content_dict.get(
            "clickThroughUrl"
        )

        link = (
            click_url.get("url")
            if click_url
            else None
        )

        provider_dict = content_dict.get(
            "provider",
            {}
        )

        provider = provider_dict.get(
            "displayName"
        )

        thumbnail = content_dict.get(
            "thumbnail"
        )

        if thumbnail and isinstance(
            thumbnail,
            dict
        ):
            image_url = thumbnail.get(
                "originalUrl"
            )
        else:
            image_url = None

        metadata = content_dict.get(
            "metadata",
            {}
        )

        is_editors_pick = metadata.get(
            "editorsPick",
            False
        )

        full_content = (
            scrape_article_content(link)
            if link
            else None
        )

        all_data.append({
            "ticker": ticker_symbol,
            "title": title,
            "published_at": published_at,
            "link": link,
            "content_type": content_type,
            "image_url": image_url,
            "is_editors_pick": is_editors_pick,
            "provider": provider,
            "publisher": provider,
            "full_content": full_content,
            "summary": summary
        })


# -----------------------------------
# Pandas DataFrame
# -----------------------------------
news_df = pd.DataFrame(all_data)

# استبدال القيم الفارغة
news_df = news_df.replace("", None)

# -----------------------------------
# Spark DataFrame
# -----------------------------------
spark_df = spark.createDataFrame(news_df)

# -----------------------------------
# Data Types + Metadata
# -----------------------------------
spark_df = (
    spark_df
    .withColumn(
        "published_at",
        expr(
            "try_to_timestamp(published_at)"
        )
    )
    .withColumn(
        "is_editors_pick",
        col(
            "is_editors_pick"
        ).cast("boolean")
    )
    .withColumn(
        "dwh_loaded_at",
        current_timestamp()
    )
)

# -----------------------------------
# Check schema
# -----------------------------------
spark_df.printSchema()

# -----------------------------------
# Save to Delta Table
# -----------------------------------


spark_df.write \
    .format("delta") \
    .mode("append") \
    .saveAsTable(
        "finance_intelligence_hub.bronze.company_news_YF_raw"
    )

In [0]:
%sql
Select * from finance_intelligence_hub.bronze.company_news_YF_raw;